# Viral Marketing Simulation

This example demonstrates how product opinions and recommendations spread through a 
**consumer social network** using `TinySocialNetwork`. Unlike a `TinyWorld` where 
everyone hears everything, this simulation shows how network topology shapes the 
diffusion of information — a critical factor for marketing strategy.

## Scenario
A health food company is launching a new organic energy drink. They want to seed 
influencers in a social network to maximize word-of-mouth. We'll compare different 
network structures to see which creates the best viral spread.

> **Note**: Results are AI-generated and for demonstration purposes only.

In [ ]:
import json
import sys
sys.path.insert(0, '..')

import tinytroupe
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld, TinySocialNetwork, TinySocialNetworkFactory
from tinytroupe.factory import TinyPersonFactory
from tinytroupe.extraction import ResultsReducer
import tinytroupe.control as control

import textwrap

## Step 1: Create the Consumer Personas

We generate a diverse set of consumer personas with varying attitudes toward 
health products, social influence tendencies, and communication styles.

In [ ]:
consumer_context = """
A diverse community of urban professionals aged 25-45 living in a mid-size American city. 
They have varied lifestyles, dietary preferences, and social media habits. Some are health 
enthusiasts, others are casual about nutrition. They form natural social clusters around 
shared interests like fitness, food, and lifestyle.
"""

factory = TinyPersonFactory(consumer_context)

# Generate diverse consumers
fitness_enthusiast = factory.generate_person(
    "A fitness enthusiast who works out daily, follows health influencers on social media, "
    "and is always looking for new health products to try. Very vocal about recommendations."
)
busy_parent = factory.generate_person(
    "A busy working parent who values convenience and practicality. Skeptical of health trends "
    "but open to recommendations from trusted friends."
)
foodie = factory.generate_person(
    "A self-described foodie who cares deeply about taste and ingredients. Active in local food "
    "communities and frequently shares food reviews with friends."
)
tech_worker = factory.generate_person(
    "A software engineer who optimizes everything including nutrition. Data-driven and reads "
    "ingredient labels carefully. Trusts scientific evidence over marketing."
)
social_butterfly = factory.generate_person(
    "A very social person who knows everyone. Not particularly health-conscious but loves "
    "trying new things and sharing experiences. Has the largest social circle."
)
yoga_instructor = factory.generate_person(
    "A yoga instructor and wellness advocate. Strongly prefers natural and organic products. "
    "Has a loyal following who trust their wellness recommendations."
)
college_student = factory.generate_person(
    "A college student on a budget who prioritizes taste and price over health benefits. "
    "Influenced by peers and social media trends."
)
retired_teacher = factory.generate_person(
    "A retired teacher who is health-conscious due to age. Values word-of-mouth "
    "recommendations from close friends over advertising."
)

all_consumers = [
    fitness_enthusiast, busy_parent, foodie, tech_worker,
    social_butterfly, yoga_instructor, college_student, retired_teacher
]

print(f"Created {len(all_consumers)} consumer personas:")
for p in all_consumers:
    print(f"  - {p.name}")

## Step 2: Build the Social Network

We'll use a **small-world network** (Watts-Strogatz model), which is the most realistic 
model for social networks — people mostly know their neighbors (local clustering), but a 
few random long-range connections create "small world" properties where everyone is just 
a few hops from everyone else.

In [ ]:
# Create a small-world social network
network = TinySocialNetworkFactory.create_small_world_network(
    "Consumer Social Network",
    all_consumers,
    k=4,      # each person connected to ~4 nearest neighbors
    p=0.2,    # 20% chance of random rewiring (creates long-range connections)
    relation_name="friends"
)

# Add some extra "influencer" connections for the social butterfly
for consumer in all_consumers:
    if consumer is not social_butterfly:
        if not network.is_in_relation_with(social_butterfly, consumer):
            network.add_relation(social_butterfly, consumer, "acquaintance")

print("=== Network Structure ===\n")
summary = network.get_network_summary()
print(f"Agents: {summary['num_agents']}")
print(f"Connections: {summary['num_edges']}")
print(f"Density: {summary['density']:.2f}")
print(f"Connected: {summary['is_connected']}")
print(f"Avg. Clustering: {summary['avg_clustering_coefficient']:.2f}")
print(f"Diameter: {summary['diameter']}")

print("\nConnections per person:")
for agent, deg in sorted(network.degree().items(), key=lambda x: -x[1]):
    print(f"  {agent.name}: {deg}")

## Step 3: Seed the Influencers

The marketing team decides to seed the **fitness enthusiast** and **yoga instructor** 
as initial product advocates. These are the most health-conscious personas who are 
most likely to authentically recommend the product.

In [ ]:
# Seed specific influencers with product knowledge
product_description = (
    "VitaBoost Organic Energy — a new organic energy drink made with green tea extract, "
    "B-vitamins, and adaptogens. No artificial sweeteners. 80mg natural caffeine. "
    "Tastes like sparkling citrus. Priced at $3.49 per can."
)

fitness_enthusiast.think(
    f"I just tried this amazing new energy drink: {product_description} "
    f"I love it! It gives me great energy for my workouts without the crash. "
    f"I should tell my friends about it."
)

yoga_instructor.think(
    f"One of my students recommended {product_description} "
    f"I appreciate that it's organic and uses adaptogens. I want to share this "
    f"with people I know who might benefit from a healthier energy option."
)

# Give everyone a shared context
network.broadcast_context_change([
    "It's a regular weekday and people are going about their normal routines. "
    "Energy and productivity are common topics of conversation."
])

# Run the simulation - let opinions spread through the network
print("Running simulation (3 rounds of conversation)...\n")
network.run(steps=3)
print("\nSimulation complete!")

## Step 4: Analyze the Viral Spread

Let's examine how the product information spread through the network.
This analysis reveals the dynamics of word-of-mouth marketing.

In [ ]:
print("=== Communication Analysis ===\n")
print(f"Total messages exchanged: {network.get_message_count()}")

print("\nMessages sent/received per person:")
for agent in sorted(network.agents, key=lambda a: -network.get_message_count(source=a)):
    sent = network.get_message_count(source=agent)
    received = network.get_message_count(target=agent)
    print(f"  {agent.name}: sent {sent}, received {received}")

# Check the degree centrality vs message volume correlation
print("\n=== Influence Analysis ===\n")
dc = network.degree_centrality()
bc = network.betweenness_centrality()

print(f"{'Person':<25} {'Degree Cent.':<15} {'Betweenness':<15} {'Msgs Sent':<10}")
print("-" * 65)
for agent in sorted(network.agents, key=lambda a: -dc[a]):
    print(f"  {agent.name:<23} {dc[agent]:<15.3f} {bc[agent]:<15.3f} {network.get_message_count(source=agent):<10}")

In [ ]:
print("=== Conversation Highlights ===\n")
print(network.pretty_current_interactions(simplified=True, max_content_length=300, last_n=3))

## Key Takeaways

This simulation demonstrates viral marketing dynamics that **require a social network 
model** (impossible with a basic `TinyWorld`):

1. **Network Topology Matters**: The small-world structure means that most people are 
   reachable within a few hops, but information still follows natural social paths.

2. **Influencer Positioning**: The fitness enthusiast and yoga instructor, seeded as 
   advocates, can only spread the word to their direct connections. The social butterfly 
   (high degree centrality) becomes a natural amplifier.

3. **Trust Cascades**: People respond differently based on who tells them about the 
   product. A recommendation from a trusted friend carries more weight than hearing 
   it from a stranger.

4. **Structural Bottlenecks**: Betweenness centrality reveals who controls information 
   flow. Seeding these "bridges" might be more effective than seeding high-degree nodes.

### Try It Yourself

- **Different network topologies**: Try `create_scale_free_network()` (power-law, like social media) 
  or `create_random_network()` to see how topology changes spread patterns
- **Different seeding strategies**: Seed the social butterfly (highest degree) instead 
  of the health-focused personas
- **More consumers**: Increase to 15-20 personas for richer dynamics
- **More steps**: Run for 5+ steps to see if opinions stabilize
- **Department network**: Try `create_department_network()` with interest-based groups